# Validation with Convolution Notebook

This notebook validates backgorund model with a non-trivial background; a powerlaw convolved with a representative low loss spectrum.

In [ ]:
from CLfitter import *
from Validation import *
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import os

## Training and plotting configuration

Set the neural-network, clustering, and output parameters used during validation.

In [ ]:

config = {
    # ── NNTrainer: energy preprocessing ──────────────────────────────────
    "log_energy":               True,
    "standardize_targets":      False,

    # ── NNTrainer.train() ────────────────────────────────────────────────
    "epochs":                   200,
    "lr":                       1e-3,
    "batch_size":               100,
    "patience":                 100,
    "min_delta":                1e-4,
    "progress":                 False,
    "lambda_deriv":             10.0,

    # ── NNTrainer.evaluate_model() ───────────────────────────────────────
    "effective_exponent_window_size": 5,

    # ── Pooler.pool_data() ───────────────────────────────────────────────
    "pool_radius":              2,
    "gaussian_kernel":          True,
    "pool_sigma":               None,   # None uses library default (radius / 2)

    # ── ClusterAnalyzer.cluster_data() ───────────────────────────────────
    "n_clusters":               6,

    # ── BackgroundTrainer.train_MC_replica_consecutive() ─────────────────
    "n_mc_replicas":            1,
    "replica_version":          "covariance",   # "triangular"|"covariance"
    "logging":                  False,

    # ── Job Info ─────────────────────────────────────────────────────────
    'parallelize':        True,
    'n_runs':             10, # Total number of replicas is n_runs * n_mc_replicas

    # ── Output paths ───────────────────────────────────────────────────
    'fitdata_path':       r'validation/fitdata',
    }

# Create output directory if it doesn't exist
os.makedirs(config['fitdata_path'], exist_ok=True)

In [ ]:
class EELSBackgroundNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.model(x)

## Generate simulated data

Build the synthetic EELS dataset with plural-scattering convolution and prepare the ground truth for later comparison.

In [ ]:
SNR = 16
pre_edge_region = 100

# def run_single_simulation(SNR, pre_edge_region):
print(f'Starting SNR={SNR} E_edge={pre_edge_region}')
E_start, E_stop = 300,600
E_edge = E_start+pre_edge_region
low_loss = LowLossEELS(E_0=200e3, beta=10e-3, dispersion=(E_stop - E_start) / 900)

SIG = SpectralImageGenerator(20, 20, 900, E_start, E_stop)

syntheticdata, energy_axis = SIG.generate_realistic_spectral_image(
    low_loss_eels=low_loss,
    apply_plural_scattering=False,
    gaussian_snr=40,
    seed=40,
    scale = 50
    )


syntheticdata = syntheticdata.reshape(900, 400)
ground_truth = SIG.background.reshape(900,400)

## Data preparation and clustering

Convert the synthetic spectral image into the format used by the fitter, then cluster the data for use in background training.

In [ ]:
handler = DataHandler()
handler.other_data(syntheticdata, np.arange(20), np.arange(20), energy_axis)
signal =  handler.signal.copy()
i = (E_start-5 ,E_edge ,E_stop+5)

range_mask = (handler.energy_axis > i[0]) & (handler.energy_axis < i[-1])  # Range for clustering shape [n_E_1]

energy_range = handler.energy_axis[range_mask] # shape [n_E_1]
signal_range = signal.copy()[range_mask,:]  # shape [n_E_1, n_y*n_x]
ground_truth_range = ground_truth.copy()[range_mask,:]
pre_edge_mask = (energy_range > i[0]) & (energy_range < i[1])  # Range for pre-edge shape [n_E_2]

clusterer = ClusterAnalyzer(signal_range)
clusterer.cluster_data(config = config, pre_edge_mask=pre_edge_mask,)
clusterer.cholesky_decomp()
plt.imshow(clusterer.clusters.reshape(20,20))  
X_builder = X_Builder(energy_range)
X_builder.prepare_X_mc_data(clusterer.cluster_centers, i[1])
X_builder.prepare_X_eval_data(clusterer.total_integrated_intensity)

## Background training routine

Train the background model with MC replicas for each cluster, then save the replicated predictions for uncertainty analysis.

In [ ]:
def train_background(ii, signal_range, pre_edge_mask, X_builder, clusterer, i):
    background_trainer = BackgroundTrainer(
        signal=signal_range,
        pre_edge_mask=pre_edge_mask,
        X_mc=X_builder.X_mc,
        X_eval=X_builder.X_eval,
        clustered_spectra_mean=clusterer.clusters_mean,
        triangular_matices=clusterer.triangular_matices,
        covariance_matrices=clusterer.clusters_covariance,
        cluster_labels=clusterer.clusters
    )

    background_trainer.train_MC_replica_consecutive(
        edge_onset=i[1],
        model = EELSBackgroundNN(),        
        config = config
    )

    np.savez(f'fitruns3/pred_{ii}.npz', pred = background_trainer.background)
    
    del background_trainer
    # Force garbage collection
    import gc
    gc.collect()

n_jobs = config['n_runs'] if config['parallelize'] else 1

from joblib import Parallel, delayed
Parallel(n_jobs=n_jobs)(
    delayed(train_background)(ii, signal_range, pre_edge_mask, X_builder, clusterer, i)
    for ii in range(0, n_jobs)
)

## Difference to theory plot and uncertainty of prediction

In [ ]:
# Load saved prediction replicas and build the full prediction array
for i in range(10):
    try:
        data1 = np.load(f'{}}/pred_{i}.npz')
        pred1 = data1['pred']
        if i == 0:
            predictions = np.zeros((10,*pred1.shape))
        predictions[i] = pred1
    except FileNotFoundError:
        print(f"File fitruns2/pred_{i}.npz not found. Skipping this replica.")
print(predictions.shape)
# reshape across all replicas
predictions = predictions.reshape(10*pred1.shape[0], pred1.shape[1], pred1.shape[2])
predictions_std = np.nanstd(predictions, axis=0).reshape(20, 20, -1)
GT = ground_truth_range.T.reshape(20, 20, 900).copy()

# normalised difference between replica ensemble and the true background
normalized_difference_to_theory = (predictions.reshape(100, 20, 20, 900) - GT[None, :, :, :]) / predictions_std[None, :, :, :]

plt.hist(normalized_difference_to_theory.flatten(), bins=50, density=True, alpha=0.6, color='g', label='Normalized Residuals')
x = np.linspace(-10, 10, 10000)
plt.plot(x, 1/np.sqrt(2*np.pi)*np.exp(-x**2/2), c='black', label='Normal Distribution')
plt.xlabel('Difference to Theory ($\sigma$)')
plt.ylabel('Probability Density')
plt.legend()
plt.xlim(-5, 5)
plt.show()

# compute cluster-level uncertainty from the fitted covariance matrices
centers = clusterer.cluster_centers
mean = clusterer.clusters_mean
cov = clusterer.clusters_covariance
labels = clusterer.clusters.reshape(20, 20)

cluster_std = np.zeros_like(predictions_std[:, :, :300])
for i in range(np.max(labels) + 1):
    mu = mean[:, i]
    sigma2 = np.diag(cov[:, :, i])
    sigma_linear = np.sqrt((np.exp(sigma2) - 1) * np.exp(2 * mu + sigma2))
    cluster_std[labels == i] = sigma_linear

# compare cluster-based uncertainties with MC replica-derived uncertainties
cluster_error_per_point = cluster_std
prediction_error_per_point = predictions_std[:, :, :300]
print(cluster_error_per_point.shape, prediction_error_per_point.shape)

x = cluster_error_per_point.flatten()
y = prediction_error_per_point.flatten()
mask = np.zeros_like(x, dtype=bool)
idx = np.random.choice(len(x), size=int(0.1 * len(x)), replace=False)
mask[idx] = True

# ---- figure and square axes ----
fig = plt.figure(figsize=(8,8))
ax = fig.add_axes([0.2,0.2,0.7,0.7])

straight_line = np.linspace(6, max(np.log(x).max(), np.log(y).max()), 100)
ax.plot(straight_line, straight_line, 'r--', label='y=x')
ax.scatter(np.log(x).flatten()[mask], np.log(y).flatten()[mask], alpha=0.1)
ax.set_ylabel('NN log-uncertainty (a.u.)', fontsize=20)
ax.set_xlabel('MC replica log-uncertainty (a.u.)', fontsize=20)
plt.tight_layout()
lgnd = plt.legend(markerscale=4)
plt.savefig('scatterplotcolored.svg')
plt.savefig('scatterplotcolored.png', dpi = 300)
plt.show()

# ─────────────────── Load and process scatter plot data ───────────────────

cluster_data = clusterer
run_data = np.load(f'{config["fitdata_path"]}/run_data.npz')
signal = run_data['signal']
labels = cluster_data['labels']
cov = cluster_data['cov']
mean = cluster_data['mean']

clusterer = ClusterAnalyzer(signal)
clusterer.cluster_data(n_clusters = mean.shape[1]+1)
centers = clusterer.cluster_centers
TII = np.log(signal.sum(axis=0))
TII_max = (TII >= centers.max())
TII_min = (TII <= centers.min())
TII_middle =  (TII <= centers.max())&(TII >= centers.min())

preds = []
for ii in range(100):
    preds.append(np.load(f'validationresults2/pred_{ii}.npz')['pred'])  
predictions = np.concatenate(preds, axis=0)
pred_median = np.median(predictions, axis=0)
lower, upper = np.percentile(predictions, [16,84], axis=0)
pred_error = (upper-lower)[:,:len(cov)]/2
print(pred_error.shape, cov.shape)
# pred_error = predictions.std(axis=0)
cluster_std = np.zeros_like(pred_error)
for i in range(np.max(labels)+1):
    mu = mean[:,i]       # shape (E,)
    sigma2 = np.diag(cov[:,:,i])     # diag of covariance, shape (E,)
    
    sigma_linear = np.sqrt( (np.exp(sigma2) - 1) * np.exp(2*mu + sigma2) )
    cluster_std[labels == i] = sigma_linear#np.sqrt(np.exp(np.diag(cov[:,:,i])))


TII_expanded = np.repeat(TII[:, None], cluster_std.shape[1], axis=1).flatten()

plt.rcParams['font.size'] = 20
# ---- figure and square axes ----
fig = plt.figure(figsize=(8,8))
ax = fig.add_axes([0.2,0.2,0.7,0.7])
# ---- scatter plot ----


sc = ax.scatter(
    cluster_std[TII_max].flatten()/cluster_std.min(),
    pred_error[TII_max].flatten()/cluster_std.min(),
    c='#FF6F61',
    s=6,
    edgecolors='none',
    label = 'Upper-Extrapolated Values'
)

sc = ax.scatter(
    cluster_std[TII_middle].flatten()/cluster_std.min(),
    pred_error[TII_middle].flatten()/cluster_std.min(),
    c='black',
    s=6,
    edgecolors='none',
    label = 'Interpolated Values'
    
)

sc = ax.scatter(
    cluster_std[TII_min].flatten()/cluster_std.min(),
    pred_error[TII_min].flatten()/cluster_std.min(),
    c='#5CECC4',
    s=6,
    edgecolors='none',
    label = 'Lower-Extrapolated Values'    
)

# --- axes & formatting ---
# ax.set_ylabel('NN uncertainty (a.u.)', fontsize=20)
# ax.set_xlabel('MC replica uncertainty (a.u.)', fontsize=20)
# ax.set_yscale('log')
# ax.set_xscale('log')

# ax.xaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, numticks=4))
# ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
# plt.setp(ax.get_xticklabels(), rotation=30, ha='right')


# 1:1 line
xmin, xmax = ax.get_xlim()
x_line = np.linspace(0, 5, 100)
ax.plot(x_line, x_line, linestyle='--', color='#FF6F61', lw = 3)

ax.set_xlim(0, 5)
ax.set_ylim(0, 10)

plt.tight_layout()
ax.set_yticks([])
ax.set_xticks([])

# lgnd = plt.legend(markerscale=4)

# for handle in lgnd.legend_handles:
#     handle.set_sizes([3

# for handle in lgnd.legend_handles:
#     handle.set_sizes([30.0])
plt.savefig('scatterplotcolored_inside.svg')
plt.savefig('scatterplotcolored_inside.png', dpi = 300)
plt.show()

# ---- figure and square axes ----
fig = plt.figure(figsize=(8,8))
ax = fig.add_axes([0.2,0.2,0.7,0.7])
# ---- scatter plot ----


sc = ax.scatter(
    cluster_std[TII_max].flatten()[1]/cluster_std.min(),
    pred_error[TII_max].flatten()[1]/cluster_std.min(),
    c='#FF6F61',
    s=6,
    edgecolors='none',
    label = 'Upper-Extrapolated Values'
)

sc = ax.scatter(
    cluster_std[TII_middle].flatten()[1]/cluster_std.min(),
    pred_error[TII_middle].flatten()[1]/cluster_std.min(),
    c='black',
    s=6,
    edgecolors='none',
    label = 'Interpolated Values'
    
)

sc = ax.scatter(
    cluster_std[TII_min].flatten()[1]/cluster_std.min(),
    pred_error[TII_min].flatten()[1]/cluster_std.min(),
    c='#5CECC4',
    s=6,
    edgecolors='none',
    label = 'Lower-Extrapolated Values'    
)

# --- axes & formatting ---
ax.set_ylabel('NN uncertainty (a.u.)', fontsize=20)
ax.set_xlabel('MC replica uncertainty (a.u.)', fontsize=20)
# ax.set_yscale('log')
# ax.set_xscale('log')

# ax.xaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, numticks=4))
# ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
# plt.setp(ax.get_xticklabels(), rotation=30, ha='right')


# 1:1 line
xmin, xmax = ax.get_xlim()
x_line = np.linspace(0, 5, 100)
ax.plot(x_line, x_line, linestyle='--', color='#FF6F61', lw = 3)

ax.set_xlim(0, 5)
ax.set_ylim(0, 10)

plt.tight_layout()
lgnd = plt.legend(markerscale=4)

# for handle in lgnd.legend_handles:
#     handle.set_sizes([30.0])
plt.savefig('scatterplotcolored_outside.svg')
plt.savefig('scatterplotcolored.png', dpi = 300)
plt.show()